# Smart IDS v3 — Notebook 02: Data Preprocessing Pipeline
Cleaning, Leakage Prevention, Splitting, Scaling, and SMOTE Rebalancing

---

### What You Will Learn
1. **Handling Zero-Division**: Converting `inf` / `-inf` to NaN and imputing with robust column medians.
2. **Preventing Target Leakage**: Dropping non-causal network identifiers (`Source IP`, `Destination IP`, `Timestamp`, etc.).
3. **Outlier Mitigation**: Clipping rate spikes at empirical $\mu \pm 5\sigma$ to stabilize neural network gradients.
4. **Stratified Splitting**: Creating a leak-free 70% Train / 15% Validation / 15% Test partition.
5. **Proper Feature Scaling**: Fitting `StandardScaler` strictly on the training set.
6. **Class Rebalancing with SMOTE**: Synthesizing minority attack samples exclusively on the training partition.

---

### Prerequisites
- Completion of Notebook 01.
- Understanding of scikit-learn transformers (`fit` vs `transform`).


## 1. Security Concept: Target Leakage in Network Security

> **Security Concept: What is Target Leakage in Cybersecurity?**
> In the CICIDS2017 testbed, attackers operated from fixed IP addresses (e.g. `192.168.10.50`), and victim servers had fixed IPs. Furthermore, certain attacks only occurred on Friday afternoon.
> 
> If we leave `Source IP`, `Destination IP`, or `Timestamp` in the feature matrix, machine learning models will simply **memorize the IP addresses and clock times**! The model will achieve 99.9% accuracy in the testbed, but when deployed into your actual company network, it will fail completely because the company's IP addresses and timestamps are different.
> 
> *Rule:* We must strictly drop all identifier columns. The model must learn the **statistical behavior** of the attack (packet lengths, inter-arrival times, flag ratios), not who sent it.


In [ ]:
import sys
from pathlib import Path

cur = Path.cwd()
root = cur if (cur / 'src').exists() else cur.parent
if str(root / 'src') not in sys.path:
    sys.path.insert(0, str(root / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

from smart_ids.config import (
    CLASS_NAMES,
    MODELS_DIR,
    PROCESSED_DATA_DIR,
    SEED,
    TEST_SIZE,
    VAL_SIZE_OF_REMAINDER,
)
from smart_ids.preprocessing import (
    clean_cicids_dataframe,
    remove_identifier_columns,
    clip_outliers,
    split_data,
    scale_features,
    apply_smote_rebalancing,
)
from smart_ids.artifacts import (
    save_scaler,
    save_label_encoder,
    save_feature_names,
)

np.random.seed(SEED)
print(f"Preprocessing environment initialized. Random seed: {SEED}")


## 2. Step 1: Cleaning Infinities, Imputing Missing Values, Deduplicating

**What we do and why:**
We replace `np.inf` and `-np.inf` with `np.nan`. If any feature has > 50% missing values, it is dropped. Remaining missing values are filled with the **feature median** (the median is resistant to extreme skew, unlike the mean). Finally, duplicate flows are removed.


In [ ]:
# Load raw dataset snapshot saved by Notebook 01
raw_csv_path = PROCESSED_DATA_DIR / 'cicids2017_raw_sample.csv'
df_raw = pd.read_csv(raw_csv_path)
print(f"Loaded raw snapshot: {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns")

# Clean infinities, NaNs, and duplicates
df_clean, audit = clean_cicids_dataframe(df_raw)
print("\n--- Data Cleaning Audit Report ---")
for k, v in audit.items():
    print(f"  {k}: {v}")


## 3. Step 2: Feature Selection (Dropping Identifiers)

**What we do and why:**
We remove columns containing keywords like `'ip'`, `'timestamp'`, `'flow id'`, `'src'`, `'dst'` to prevent target leakage.


In [ ]:
df_features, dropped_cols = remove_identifier_columns(df_clean)
print(f"Dropped {len(dropped_cols)} identifier columns: {dropped_cols}")
print(f"Remaining feature matrix shape: {df_features.shape}")


## 4. Step 3: Outlier Clipping ($\mu \pm 5\sigma$)

**What we do and why:**
In network telemetry, anomalous network glitches can produce rate values like $10^{12}$ bytes/sec. These massive numbers cause exploding gradients during neural network training. We clip all features to $[\mu - 5\sigma, \mu + 5\sigma]$.


In [ ]:
df_clipped = clip_outliers(df_features, sigma_multiple=5.0)
print("Outlier clipping complete. Feature values are now bounded within +/- 5 standard deviations.")


## 5. Step 4: Stratified Train / Validation / Test Splitting

> **Common Mistake: Data Leakage Before Splitting**
> A very common beginner mistake is to fit a scaler or run SMOTE on the entire dataset *before* creating train and test sets. This leaks future test information into the training data and produces invalid, overly optimistic test scores!
> 
> *The Golden Rule:* Split first. Fit transformers and oversamplers **on the training partition ONLY**.

We perform a stratified split:
- **Test Set**: 15% of the total dataset.
- **Validation Set**: 15% of the remainder (~12.75% of total).
- **Train Set**: The remaining ~72.25%.


In [ ]:
X_train_df, X_val_df, X_test_df, y_train_raw, y_val_raw, y_test_raw = split_data(
    df_clipped,
    target_col='Label',
    test_size=TEST_SIZE,
    val_size_of_remainder=VAL_SIZE_OF_REMAINDER,
    random_state=SEED,
)

# Encode categorical labels to integers 0..7
label_encoder = LabelEncoder()
label_encoder.fit(CLASS_NAMES)

y_train = label_encoder.transform(y_train_raw)
y_val = label_encoder.transform(y_val_raw)
y_test = label_encoder.transform(y_test_raw)

feature_names = list(X_train_df.columns)
print(f"Feature count: {len(feature_names)}")
print(f"Classes encoded: {list(zip(range(len(CLASS_NAMES)), CLASS_NAMES))}")


## 6. Step 5: Feature Standardization (Fit on Train Only)

**What we do and why:**
We fit `StandardScaler` strictly on `X_train_df`, then transform `X_val_df` and `X_test_df`.


In [ ]:
X_train_scaled, X_val_scaled, X_test_scaled, scaler = scale_features(
    X_train_df, X_val_df, X_test_df
)

print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_val_scaled shape:   {X_val_scaled.shape}")
print(f"X_test_scaled shape:  {X_test_scaled.shape}")
print(f"Train Mean (first 3 features): {X_train_scaled[:, :3].mean(axis=0).round(4)}")
print(f"Train Std (first 3 features):  {X_train_scaled[:, :3].std(axis=0).round(4)}")


## 7. Step 6: SMOTE Class Rebalancing (Train Only)

> **Common Mistake: Applying SMOTE to Validation or Test Data**
> Validation and test sets must represent the **real world**, which is naturally imbalanced! If you apply SMOTE to validation or test data, you are testing your model on synthetic data that doesn't exist in reality.
> 
> Apply SMOTE **exclusively to the training partition**.

We use a dynamic $k$-neighbor guard: $k = \max(1, \min(5, 	ext{smallest\_class\_size} - 1))$.


In [ ]:
# Pre-SMOTE distribution
pre_counts = pd.Series(y_train).value_counts().sort_index()

# Apply SMOTE exclusively on training partition
X_train_sm, y_train_sm = apply_smote_rebalancing(
    X_train_scaled, y_train, random_state=SEED
)

# Post-SMOTE distribution
post_counts = pd.Series(y_train_sm).value_counts().sort_index()

smote_comparison = pd.DataFrame({
    'Class Name': CLASS_NAMES,
    'Pre-SMOTE Train': pre_counts.values,
    'Post-SMOTE Train': post_counts.values,
})
print("--- Training Partition SMOTE Rebalancing ---")
print(smote_comparison)


## 8. Persisting Preprocessing Artifacts

We save the fitted `StandardScaler`, `LabelEncoder`, feature name list, and preprocessed NumPy arrays so downstream notebooks can load them directly.


In [ ]:
save_scaler(scaler)
save_label_encoder(label_encoder)
save_feature_names(feature_names)

# Save processed arrays for instant loading in Notebooks 03, 04, 05, 06
np.save(PROCESSED_DATA_DIR / 'X_train_sm.npy', X_train_sm)
np.save(PROCESSED_DATA_DIR / 'y_train_sm.npy', y_train_sm)
np.save(PROCESSED_DATA_DIR / 'X_train_scaled.npy', X_train_scaled)
np.save(PROCESSED_DATA_DIR / 'y_train.npy', y_train)
np.save(PROCESSED_DATA_DIR / 'X_val_scaled.npy', X_val_scaled)
np.save(PROCESSED_DATA_DIR / 'y_val.npy', y_val)
np.save(PROCESSED_DATA_DIR / 'X_test_scaled.npy', X_test_scaled)
np.save(PROCESSED_DATA_DIR / 'y_test.npy', y_test)

print("All preprocessing artifacts and NumPy splits saved successfully!")


---
## Summary and Next Notebook

In this notebook, we built a leak-free data engineering pipeline:
- Converted zero-division infinities to NaN and median-imputed them.
- Excluded IP and timestamp columns to eliminate target leakage.
- Clipped features at $\pm 5\sigma$ to protect gradient descent.
- Split data into train (70%), validation (15%), and test (15%).
- Fit `StandardScaler` and `SMOTE` **on the training data only**.

**Next up:** In **Notebook 03: Supervised Multi-Class Models**, we will train our XGBoost baseline, deep Multi-Layer Perceptron (MLP), and 1D-Convolutional Neural Network (1D-CNN)!
